In [1]:
import requests
import boto3
import os
import json
from urllib.parse import urlparse, urljoin
from datetime import datetime, timezone, timedelta

# --- Configuration ---
# MinIO configuration (read from environment variables set in docker-compose)
MINIO_URL = os.environ.get('MINIO_URL')
MINIO_ACCESS_KEY = os.environ.get('MINIO_USER')
MINIO_SECRET_KEY = os.environ.get('MINIO_PASSWORD')
MINIO_BUCKET_NAME = os.environ.get('MINIO_BUCKET_NAME')

# Label Studio configuration (read from environment variables set in docker-compose)
LABEL_STUDIO_URL = os.environ.get('LABEL_STUDIO_URL')
LABEL_STUDIO_TOKEN = os.environ.get('LABEL_STUDIO_USER_TOKEN')

# Project ID - Get this manually from Label Studio UI after creating the project
PROJECT_ID = 1

# Data filtering options
RECENT_TIME_THRESHOLD = datetime.now(timezone.utc) - timedelta(days=7) # Example: last 7 days

# --- Initialize Clients ---
s3 = None
try:
    s3 = boto3.client(
        "s3",
        endpoint_url=MINIO_URL,
        aws_access_key_id=MINIO_ACCESS_KEY,
        aws_secret_access_key=MINIO_SECRET_KEY,
        region_name='us-east-1'
    )
    s3.list_buckets()
    print("MinIO client initialized successfully.")
except Exception as e:
    print(f"Error initializing MinIO client: {e}")
    s3 = None

print(f"Using Label Studio URL: {LABEL_STUDIO_URL}")
print(f"Using Label Studio Project ID: {PROJECT_ID}")

MinIO client initialized successfully.
Using Label Studio URL: http://label-studio:8080
Using Label Studio Project ID: 1


In [2]:
def get_minio_object_content(s3_client, bucket, key):
    """Helper to get content from an S3 object."""
    if not s3_client or not key:
        return None
    try:
        response = s3_client.get_object(Bucket=bucket, Key=key)
        return response['Body'].read().decode('utf-8')
    except s3_client.exceptions.ClientError as e:
        if e.response['Error']['Code'] == 'NoSuchKey':
             # print(f"Warning: Object not found s3://{bucket}/{key}")
             return None
        else:
             print(f"Error getting object s3://{bucket}/{key}: {e}")
             return None
    except Exception as e:
        print(f"An unexpected error occurred getting object s3://{bucket}/{key}: {e}")
        return None

In [3]:
tasks_to_import = []
processed_metadata_count = 0
tasks_generated_count = 0

if s3:
    print(f"Listing metadata objects in bucket '{MINIO_BUCKET_NAME}' with prefix 'metadata/'...")
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=MINIO_BUCKET_NAME, Prefix="metadata/"):
        for obj in page.get("Contents", []):
            metadata_key = obj["Key"]
            if not metadata_key.endswith(".json"):
                 continue

            processed_metadata_count += 1
            # print(f"Processing metadata object {processed_metadata_count}: {metadata_key}")

            metadata_content_str = get_minio_object_content(s3, MINIO_BUCKET_NAME, metadata_key)

            if metadata_content_str:
                try:
                    metadata = json.loads(metadata_content_str)

                    # --- Filtering Logic (Customize as needed, same as before) ---
                    timestamp_str = metadata.get("timestamp_utc")
                    if timestamp_str:
                         try:
                             timestamp = datetime.fromisoformat(timestamp_str.replace('Z', '+00:00'))
                             if timestamp < RECENT_TIME_THRESHOLD:
                                 continue
                         except ValueError:
                             print(f"Warning: Could not parse timestamp for {metadata_key}: {timestamp_str}")
                             pass

                    # Add other filters here if needed (e.g., by processing_task, user_rating_helpful)
                    # --- End Filtering Logic ---

                    # Fetch related data (transcript and output text)
                    transcript_key = metadata.get("transcript_s3_key")
                    output_key = metadata.get("output_s3_key")
                    query_text = metadata.get("query", "")

                    transcript_text = get_minio_object_content(s3, MINIO_BUCKET_NAME, transcript_key)
                    if transcript_text is None:
                         print(f"Warning: Could not fetch transcript text for {metadata_key}. Skipping task.")
                         continue

                    model_output_text = get_minio_object_content(s3, MINIO_BUCKET_NAME, output_key)
                    if model_output_text is None:
                         print(f"Warning: Could not fetch model output text for {metadata_key}. Skipping task.")
                         continue

                    # Create a task dictionary for Label Studio (keys match template $variables)
                    task_data = {
                        "transcript": transcript_text,
                        "query": query_text,
                        "model_output": model_output_text,
                        "filename": metadata.get("original_filename", ""),
                        "interaction_id": metadata.get("processing_uuid", metadata_key.split('/')[1] if len(metadata_key.split('/')) > 1 else "N/A")
                    }

                    tasks_to_import.append({
                        "data": task_data,
                        "meta": {"processing_uuid": task_data["interaction_id"], "metadata_key": metadata_key}
                    })
                    tasks_generated_count += 1

                except json.JSONDecodeError:
                    print(f"Error decoding JSON for {metadata_key}")
                except Exception as e:
                    print(f"An error processing metadata object {metadata_key}: {e}")

print(f"Generated {len(tasks_to_import)} tasks from {processed_metadata_count} metadata files.")

Listing metadata objects in bucket 'production' with prefix 'metadata/'...
Generated 4 tasks from 4 metadata files.


In [4]:
if tasks_to_import:
    print(f"Importing {len(tasks_to_import)} tasks into Label Studio project ID {PROJECT_ID} at {LABEL_STUDIO_URL}...")
    import_url = f"{LABEL_STUDIO_URL}/api/projects/{PROJECT_ID}/import"
    headers = {"Authorization": f"Token {LABEL_STUDIO_TOKEN}"}

    try:
        response = requests.post(import_url, json=tasks_to_import, headers=headers)

        if response.status_code == 201:
            print(f"Successfully imported {len(tasks_to_import)} tasks.")
        else:
            print(f"Failed to import tasks. Status Code: {response.status_code}")
            print("Response:", response.text)
            response.raise_for_status()

    except requests.exceptions.RequestException as e:
        print(f"Error making request to Label Studio API at {import_url}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred during import: {e}")
else:
    print("No tasks to import.")

Importing 4 tasks into Label Studio project ID 1 at http://label-studio:8080...
Failed to import tasks. Status Code: 401
Response: {"id":"ba6cb075-bae8-4206-bf96-e27ad0013646","status_code":401,"version":"1.17.0","detail":"Authentication token no longer valid: legacy token authentication has been disabled for this organization","exc_info":null}
Error making request to Label Studio API at http://label-studio:8080/api/projects/1/import: 401 Client Error: Unauthorized for url: http://label-studio:8080/api/projects/1/import
